# Daily Practice — 2026-11-11 — ML Testing & MLOps: Building a Data Contract Test Suite

**Dataset:** [Open Exoplanet Catalogue, via seaborn-data `planets.csv`](https://github.com/mwaskom/seaborn-data) —
1035 real confirmed exoplanet discovery records (detection `method`, `number` of planets found in
that system, `orbital_period` in days, `mass` in Jupiter masses, `distance` to the star in parsecs,
discovery `year`), fetched directly from the raw GitHub CSV at notebook build time. The dataset has
genuine real-world missingness (`mass` is unmeasured for ~50% of systems, `distance` for ~22%) —
nothing here is faked.

## Problem statement

You're the MLOps/QA engineer on a team that retrains an exoplanet-mass regression model every week
on a fresh batch pulled from an upstream ingestion job. Upstream has broken before, silently:
a unit-conversion bug turned some masses negative, a new detection method got added to the raw feed
before anyone updated the allow-list, and once a bad JOIN duplicated a third of the batch. None of
these would show up as a Python exception — the pipeline would happily train on garbage and ship a
model nobody can trust.

Your job is to build a **data contract test suite**: a function that checks an incoming batch
against an explicit, versioned contract (required columns, types, missingness ceilings, numeric
ranges, allowed categories, duplicate rate, minimum row count) *before* training is allowed to
start, plus a battery of tests proving the suite actually catches each real failure mode — not just
a single happy-path demo.

**What you should produce, concretely:**
1. `validate_schema(df, contract)` — returns a report dict `{"passed": bool, "violations": [...],
   "n_rows": int}` listing every violation found (not just the first one).
2. `training_gate(df, contract)` — calls `validate_schema` and returns `(should_train, report)`.
3. A test battery of **6 functions**, each injecting exactly one convincing failure mode into a copy
   of the reference data and asserting the gate catches *that specific* violation (not just that it
   fails). One test should assert the clean reference batch passes.
4. A "this week's ingestion" batch that combines 2–3 real failure modes at once, run through the
   gate, with its violation list printed — this is the scenario the suite exists to catch.

Injecting specific corruptions into copies of the real data is the one synthetic part of this
exercise, and it's used deliberately: to prove the validator catches *each* controlled failure mode,
you need to be able to inject it on demand. The data you validate is always real.

## Setup

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/planets.csv"


## Load the reference batch

This is the "known-good" batch: last week's successfully-ingested, already-trained-on data. It's
what new batches get compared against.

In [ ]:
df_reference = pd.read_csv(DATA_URL)
print(df_reference.shape)
df_reference.head()


In [ ]:
df_reference.isna().mean().round(3)


## The contract

A contract is just a dict of rules. It's deliberately explicit and inspectable — anyone on the team
should be able to read it and know exactly what's being enforced, without reading the validator code.

This one is given to you. Note `allowed_categories["method"]` is derived from the reference batch
itself (today's known detection methods) — a genuinely new method next year should trip the gate
until a human reviews and updates the contract.

In [ ]:
CONTRACT = {
    "required_columns": ["method", "number", "orbital_period", "mass", "distance", "year"],
    "dtype_kinds": {
        "method": "string",      # not pd.api.types.is_numeric_dtype
        "number": "integer",
        "orbital_period": "float",
        "mass": "float",
        "distance": "float",
        "year": "integer",
    },
    "max_missing_frac": {
        "orbital_period": 0.10,
        "mass": 0.55,
        "distance": 0.30,
    },
    "numeric_ranges": {
        "orbital_period": (0, None),
        "mass": (0, None),
        "distance": (0, None),
        "year": (1989, 2026),
        "number": (1, None),
    },
    "allowed_categories": {
        "method": set(df_reference["method"].unique()),
    },
    "min_rows": 200,
    "max_duplicate_frac": 0.02,
}


## Your task

Fill in the TODOs below. Signatures are given — don't change them, the test battery and solution
assume these names and return shapes.

`validate_schema` must check, in this order, and **accumulate every violation it finds** (don't
return early on the first one — an engineer debugging a broken pipeline needs the full list):

1. Every column in `contract["required_columns"]` is present.
2. For each present column with an entry in `contract["dtype_kinds"]`, the dtype matches the
   expected kind (`"integer"`, `"float"`, or `"string"` — use `pd.api.types.is_integer_dtype`,
   `is_float_dtype`, and "not `is_numeric_dtype`" respectively; don't compare `str(dtype)` directly,
   pandas string-dtype reporting isn't stable across versions).
3. For each column in `contract["max_missing_frac"]`, the fraction of nulls doesn't exceed the
   ceiling.
4. For each column in `contract["numeric_ranges"]`, no non-null value falls outside `(lo, hi)`
   (either bound may be `None`, meaning unbounded on that side).
5. For each column in `contract["allowed_categories"]`, no value outside the allowed set appears.
6. The row count is at least `contract["min_rows"]`.
7. The fraction of fully-duplicated rows doesn't exceed `contract["max_duplicate_frac"]`.

Violation strings should be identifiable by a stable prefix per check type (e.g.
`"range_violation_low:mass:min_seen=-1.5"`) so the tests below can assert on substrings.

In [ ]:
def validate_schema(df, contract):
    """Return {"passed": bool, "violations": list[str], "n_rows": int}."""
    violations = []

    # TODO 1: required columns present
    ...

    # TODO 2: dtype kinds (only check columns that are actually present)
    ...

    # TODO 3: missingness ceilings
    ...

    # TODO 4: numeric ranges
    ...

    # TODO 5: allowed categories
    ...

    # TODO 6: minimum row count
    ...

    # TODO 7: duplicate row fraction
    ...

    return {"passed": len(violations) == 0, "violations": violations, "n_rows": len(df)}


def training_gate(df, contract):
    """Return (should_train: bool, report: dict)."""
    # TODO: call validate_schema and unpack the result
    ...


### Test battery

Each test builds its *own* corrupted copy of `df_reference` — never mutate `df_reference` itself.
Each one injects exactly one failure mode and asserts the gate both (a) fails and (b) the violation
list contains the specific check you expect, by substring. The first test asserts the opposite: that
clean reference data passes untouched.

In [ ]:
def test_clean_reference_passes():
    # TODO: assert training_gate(df_reference, CONTRACT) passes with zero violations
    ...


def test_catches_negative_mass():
    # TODO: copy df_reference, set a few `mass` values negative, assert the gate fails
    # and a "range_violation_low:mass" violation is present
    ...


def test_catches_unseen_category():
    # TODO: copy df_reference, set one `method` value to something not in CONTRACT's allow-list,
    # assert the gate fails with an "unexpected_categories:method" violation
    ...


def test_catches_duplicate_flood():
    # TODO: build a batch that concatenates df_reference with a large slice of itself, assert
    # the gate fails with a "duplicate_rows_exceeded" violation
    ...


def test_catches_missingness_spike():
    # TODO: copy df_reference, null out ~95% of `distance` (use RANDOM_STATE), assert the gate
    # fails with a "missingness_exceeded:distance" violation
    ...


def test_catches_empty_batch():
    # TODO: pass an empty (zero-row) dataframe with the right columns, assert the gate fails
    # with a "row_count_too_low" violation
    ...


for test_fn in [
    test_clean_reference_passes,
    test_catches_negative_mass,
    test_catches_unseen_category,
    test_catches_duplicate_flood,
    test_catches_missingness_spike,
    test_catches_empty_batch,
]:
    test_fn()
    print("PASS", test_fn.__name__)


### This week's ingestion batch

TODO: build `this_week`, a 250-row sample of `df_reference` (use `RANDOM_STATE`) with **three**
failure modes stacked on top of each other: a handful of negative `mass` values, one unseen
`method` value, and ~95% of `distance` nulled out. Run it through `training_gate` and print whether
training should proceed and the full violation list. Then run the clean `df_reference` through the
same gate for comparison.

In [ ]:
# TODO: build this_week with the three stacked failure modes, then:
# passed, report = training_gate(this_week, CONTRACT)
# print("this_week -> should_train:", passed)
# print(report["violations"])
#
# passed_ref, report_ref = training_gate(df_reference, CONTRACT)
# print("reference -> should_train:", passed_ref, report_ref["violations"])
...


### Write-up

TODO: in 3–5 sentences — if this gate had been wired into the real pipeline *before* the week the
mass-sign bug shipped, what would have happened differently, and why is "the model's test-set R²
looked fine last week" not a substitute for a contract check like this one? Also: `allowed_categories`
for `method` was derived from the reference batch itself. What's the tradeoff of that choice versus
hand-maintaining a fixed allow-list — what failure mode does each approach catch that the other
misses?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def validate_schema(df, contract):
    violations = []

    for col in contract["required_columns"]:
        if col not in df.columns:
            violations.append(f"missing_column:{col}")

    present_cols = [c for c in contract["required_columns"] if c in df.columns]

    kind_checks = {
        "integer": pd.api.types.is_integer_dtype,
        "float": pd.api.types.is_float_dtype,
        "string": lambda s: not pd.api.types.is_numeric_dtype(s),
    }
    for col, expected_kind in contract["dtype_kinds"].items():
        if col in present_cols and not kind_checks[expected_kind](df[col]):
            violations.append(
                f"dtype_mismatch:{col}:expected_kind={expected_kind}:actual={df[col].dtype}"
            )

    for col, max_frac in contract["max_missing_frac"].items():
        if col in present_cols:
            frac = df[col].isna().mean()
            if frac > max_frac:
                violations.append(f"missingness_exceeded:{col}:{frac:.3f}>{max_frac}")

    for col, (lo, hi) in contract["numeric_ranges"].items():
        if col in present_cols:
            values = df[col].dropna()
            if lo is not None and (values < lo).any():
                violations.append(f"range_violation_low:{col}:min_seen={values.min()}")
            if hi is not None and (values > hi).any():
                violations.append(f"range_violation_high:{col}:max_seen={values.max()}")

    for col, allowed in contract["allowed_categories"].items():
        if col in present_cols:
            unexpected = set(df[col].dropna().unique()) - allowed
            if unexpected:
                violations.append(f"unexpected_categories:{col}:{sorted(unexpected)}")

    n_rows = len(df)
    if n_rows < contract["min_rows"]:
        violations.append(f"row_count_too_low:{n_rows}<{contract['min_rows']}")

    if n_rows > 0:
        dup_frac = df.duplicated().mean()
        if dup_frac > contract["max_duplicate_frac"]:
            violations.append(f"duplicate_rows_exceeded:{dup_frac:.3f}>{contract['max_duplicate_frac']}")

    return {"passed": len(violations) == 0, "violations": violations, "n_rows": n_rows}


def training_gate(df, contract):
    report = validate_schema(df, contract)
    return report["passed"], report
```

```python
def test_clean_reference_passes():
    passed, report = training_gate(df_reference, CONTRACT)
    assert passed, report["violations"]


def test_catches_negative_mass():
    bad = df_reference.copy()
    bad.loc[bad.index[:5], "mass"] = -3.0
    passed, report = training_gate(bad, CONTRACT)
    assert not passed
    assert any("range_violation_low:mass" in v for v in report["violations"])


def test_catches_unseen_category():
    bad = df_reference.copy()
    bad.loc[bad.index[0], "method"] = "Quantum Entanglement Timing"
    passed, report = training_gate(bad, CONTRACT)
    assert not passed
    assert any("unexpected_categories:method" in v for v in report["violations"])


def test_catches_duplicate_flood():
    bad = pd.concat([df_reference, df_reference.iloc[:400]], ignore_index=True)
    passed, report = training_gate(bad, CONTRACT)
    assert not passed
    assert any("duplicate_rows_exceeded" in v for v in report["violations"])


def test_catches_missingness_spike():
    bad = df_reference.copy()
    rng = np.random.RandomState(RANDOM_STATE)
    mask = rng.rand(len(bad)) < 0.95
    bad.loc[mask, "distance"] = np.nan
    passed, report = training_gate(bad, CONTRACT)
    assert not passed
    assert any("missingness_exceeded:distance" in v for v in report["violations"])


def test_catches_empty_batch():
    bad = df_reference.iloc[0:0].copy()
    passed, report = training_gate(bad, CONTRACT)
    assert not passed
    assert any("row_count_too_low" in v for v in report["violations"])
```

```python
rng = np.random.RandomState(RANDOM_STATE)
this_week = df_reference.sample(n=250, random_state=RANDOM_STATE).reset_index(drop=True)
this_week.loc[this_week.index[:10], "mass"] = -1.5
this_week.loc[this_week.index[20], "method"] = "Unknown Method X"
mask = rng.rand(len(this_week)) < 0.95
this_week.loc[mask, "distance"] = np.nan

passed, report = training_gate(this_week, CONTRACT)
print("this_week -> should_train:", passed)
print(report["violations"])
# this_week -> should_train: False
# ['missingness_exceeded:distance:0.968>0.3',
#  'range_violation_low:mass:min_seen=-1.5',
#  "unexpected_categories:method:['Unknown Method X']"]

passed_ref, report_ref = training_gate(df_reference, CONTRACT)
print("reference -> should_train:", passed_ref, report_ref["violations"])
# reference -> should_train: True []
```

**Write-up pointers:** the gate would have blocked the mass-sign-bug batch outright, before a
single training run — the failure is caught at ingestion, not three weeks later when someone
notices production predictions look wrong. A held-out R² looking fine is a property of *the model
on last week's data*; it tells you nothing about whether *this week's* batch is the kind of data the
model was ever validated against — R² can't see a sign flip in an input feature the model wasn't
using yet, or a duplicate flood that just makes a metric look artificially stable. Deriving
`allowed_categories` from the reference batch catches genuinely novel categories immediately with
zero maintenance burden, but it also means a *typo* that happens to coincide with a legitimate rare
method already seen once would slip through; a hand-maintained allow-list catches typos reliably but
needs a human to update it every time astronomy adds a real new detection method, or it starts
rejecting good data.

</details>